# Medical LLM Notebook 04 — Graph Feature Analysis

Compute and validate the graph-theoretic features used in the manuscript on the pilot word-level attention graphs generated by Notebook 03.

The notebook implements normalized degree, PageRank, weighted betweenness, strength, GCC, corrected GCCDC, communication efficiency, and perturbation-based Structural Influence. It is a pilot validation notebook; Notebook 05 applies the same definitions automatically to all 100 pairs.

In [ ]:
!pip -q install networkx pandas numpy

In [ ]:
import os
import glob
import json
import zipfile
import shutil
import numpy as np
import pandas as pd
import networkx as nx
from google.colab import files

EPS = 1e-12


## 1. Upload Notebook 03 output ZIP

In [ ]:
uploaded = files.upload()
zip_files = [x for x in uploaded if x.lower().endswith(".zip")]
assert zip_files, "Upload Notebook3_for_Notebook4.zip"

WORK = "/content/Notebook04_input"
os.makedirs(WORK, exist_ok=True)

with zipfile.ZipFile(zip_files[0], "r") as z:
    z.extractall(WORK)

print("Extracted files:")
for p in glob.glob(WORK + "/**/*", recursive=True):
    if os.path.isfile(p):
        print(p)


In [ ]:
def one(pattern):
    matches = glob.glob(WORK + "/**/" + pattern, recursive=True)
    assert matches, f"Missing {pattern}"
    return matches[0]

F_GRAPH = one("MEDH096_factual_word_graph.graphml")
H_GRAPH = one("MEDH096_hallucinated_word_graph.graphml")
QAS_FILE = one("MEDH096_QAS.csv")

Gf = nx.read_graphml(F_GRAPH, node_type=int)
Gh = nx.read_graphml(H_GRAPH, node_type=int)
qas = pd.read_csv(QAS_FILE)

for G in [Gf, Gh]:
    for _, _, d in G.edges(data=True):
        d["weight"] = float(d["weight"])
        d["distance"] = float(d["distance"])

print("Factual:", Gf.number_of_nodes(), "nodes,", Gf.number_of_edges(), "edges")
print("Hallucinated:", Gh.number_of_nodes(), "nodes,", Gh.number_of_edges(), "edges")
display(qas)


## 2. Graph measures

In [ ]:
def normalized_degree(G):
    n = G.number_of_nodes()
    if n <= 1:
        return {v: 0.0 for v in G.nodes()}
    return {
        v: (G.in_degree(v) + G.out_degree(v)) / (2.0 * (n - 1))
        for v in G.nodes()
    }


def graph_gcc(G):
    if G.number_of_nodes() < 3:
        return 0.0
    return float(nx.transitivity(G.to_undirected()))


def communication_efficiency(G):
    n = G.number_of_nodes()
    if n <= 1:
        return 0.0

    total = 0.0
    for source, lengths in nx.all_pairs_dijkstra_path_length(
        G,
        weight="distance"
    ):
        for target, d in lengths.items():
            if source != target and np.isfinite(d) and d > 0:
                total += 1.0 / d

    return total / (n * (n - 1))


def structural_influence(G):
    phi0 = communication_efficiency(G)

    if phi0 <= EPS:
        return {v: 0.0 for v in G.nodes()}

    scores = {}

    for v in G.nodes():
        perturbed = G.copy()

        incident = list(perturbed.in_edges(v)) + list(perturbed.out_edges(v))
        perturbed.remove_edges_from(incident)

        phi_v = communication_efficiency(perturbed)

        scores[v] = (phi0 - phi_v) / phi0

    return scores


In [ ]:
def centrality_table(G):
    degree = normalized_degree(G)
    gcc = graph_gcc(G)

    pagerank = nx.pagerank(
        G,
        weight="weight"
    )

    betweenness = nx.betweenness_centrality(
        G,
        weight="distance",
        normalized=True
    )

    si = structural_influence(G)

    rows = []

    for v in G.nodes():
        in_strength = sum(
            d["weight"]
            for _, _, d in G.in_edges(v, data=True)
        )
        out_strength = sum(
            d["weight"]
            for _, _, d in G.out_edges(v, data=True)
        )

        rows.append({
            "Node": v,
            "Word": G.nodes[v].get("word", str(v)),
            "Degree": degree[v],
            "PageRank": pagerank[v],
            "Betweenness": betweenness[v],
            "In_Strength": in_strength,
            "Out_Strength": out_strength,
            "GCCDC": degree[v] * (1.0 - gcc),
            "Structural_Influence": si[v]
        })

    return pd.DataFrame(rows)


def response_features(G, qas_row):
    c = centrality_table(G)
    n = G.number_of_nodes()
    weights = [d["weight"] for _, _, d in G.edges(data=True)]

    return {
        "Words": n,
        "Edges": G.number_of_edges(),
        "Density": nx.density(G),
        "GCC": graph_gcc(G),
        "Communication_Efficiency": communication_efficiency(G),
        "Mean_Edge_Weight": float(np.mean(weights)) if weights else 0.0,
        "Max_PageRank": float(c["PageRank"].max()),
        "PageRank_Concentration": float(n * c["PageRank"].max()) if n else 0.0,
        "Mean_Betweenness": float(c["Betweenness"].mean()),
        "Max_Betweenness": float(c["Betweenness"].max()),
        "Max_GCCDC": float(c["GCCDC"].max()),
        "Mean_Structural_Influence": float(c["Structural_Influence"].mean()),
        "Max_Structural_Influence": float(c["Structural_Influence"].max()),
        "QAS_Mean": float(qas_row["QAS_Mean"]),
        "QAS_Median": float(qas_row["QAS_Median"]),
        "QAS_Std": float(qas_row["QAS_Std"])
    }, c


## 3. Pilot factual–hallucinated comparison

In [ ]:
def get_qas(response_type):
    row = qas[qas["Response_Type"].str.lower() == response_type.lower()]
    assert len(row) == 1
    return row.iloc[0]

f_features, f_nodes = response_features(
    Gf,
    get_qas("Factual")
)

h_features, h_nodes = response_features(
    Gh,
    get_qas("Hallucinated")
)

comparison = pd.DataFrame(
    {
        "Factual": f_features,
        "Hallucinated": h_features
    }
)

comparison["Delta_H_minus_F"] = (
    comparison["Hallucinated"]
    -
    comparison["Factual"]
)

display(comparison)


In [ ]:
print("Top factual Structural Influence:")
display(
    f_nodes.sort_values(
        "Structural_Influence",
        ascending=False
    ).head(10)
)

print("Top hallucinated Structural Influence:")
display(
    h_nodes.sort_values(
        "Structural_Influence",
        ascending=False
    ).head(10)
)


## 4. Save pilot graph-analysis outputs

In [ ]:
OUT = "/content/Notebook04_outputs"
os.makedirs(OUT, exist_ok=True)

comparison.to_csv(
    f"{OUT}/MEDH096_Response_Feature_Comparison.csv"
)

f_nodes.to_csv(
    f"{OUT}/MEDH096_Factual_Node_Centralities.csv",
    index=False
)

h_nodes.to_csv(
    f"{OUT}/MEDH096_Hallucinated_Node_Centralities.csv",
    index=False
)

archive = shutil.make_archive(
    "/content/Notebook04_Graph_Feature_Outputs",
    "zip",
    OUT
)

print("Created:", archive)
files.download(archive)
